        # 🎲 L02　機率與分布
        **統計冒險之旅 2026**　｜　Day 1（09/21 一）🌄 統計之丘　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch2 觀念；資料：勇者咖啡八月銷售


        ### 🎯 這一關你會學到
        - 機率＝長期頻率：用模擬算機率
- 常態分布、68–95–99.7 與 z 分數（標準化）
- 二項分布：幾個客人會買甜點

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L02"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["2-1", "2-2", "2-3", "2-4", "2-5"]
_XP_EACH = 20
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_2_1(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "六點機率"), 1/6, 0.015): return (False, "六點機率 = (骰子 == 6).mean()，應該接近 0.1667。")
    return (約等於(抓變數(ns, "和為7機率"), 1/6, 0.015), "和為7機率 = ((a + b) == 7).mean()，應該接近 0.1667。")
任務定義("2-1", _check_2_1, 提示="布林陣列取 .mean() 就是「比例」。")

def _check_2_2(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "小於340機率"), 0.105650, 0.002): return (False, "小於340機率 = stats.norm.cdf(340, 350, 8)。")
    if not 約等於(抓變數(ns, "介於340到360機率"), 0.788700, 0.002): return (False, "介於 = cdf(360) - cdf(340)。")
    return (約等於(抓變數(ns, "前百分之五門檻"), 363.1588, 0.05), "前百分之五門檻 = stats.norm.ppf(0.95, 350, 8)。")
任務定義("2-2", _check_2_2, 提示="cdf 是「小於」的機率；「介於」用兩個 cdf 相減；ppf 是 cdf 的反函數。")

def _check_2_3(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "金額500的z"), 3.84284, 0.01): return (False, "z = (500 - 平均) / 標準差。")
    if not 約等於(抓變數(ns, "標準化平均"), 0, 1e-6): return (False, "標準化後平均應該是 0（誤差極小）。")
    return (約等於(抓變數(ns, "標準化標準差"), 1, 1e-6), "標準化後標準差應該是 1。")
任務定義("2-3", _check_2_3, 提示="金額z = (df['金額'] - 平均) / 標準差")

def _check_2_4(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "恰好3人"), 0.266828, 0.002): return (False, "恰好3人 = stats.binom.pmf(3, 10, 0.3)。")
    if not 約等於(抓變數(ns, "至少5人"), 0.150268, 0.002): return (False, "至少5人 = 1 - stats.binom.cdf(4, 10, 0.3)。")
    return (約等於(抓變數(ns, "期望人數"), 3, 1e-6), "期望人數 = n * p = 3。")
任務定義("2-4", _check_2_4, 提示="「至少 5 人」= 1 −「最多 4 人」。")

def _check_2_5(run):
    out, ns = run()
    if not run.figs: return (False, "沒有畫出圖。")
    if "常態" not in " ".join(f["title"] for f in run.figs): return (False, "標題要包含「常態」。")
    return (約等於(抓變數(ns, "下界"), 342, 1e-6) and 約等於(抓變數(ns, "上界"), 358, 1e-6), "68% 範圍是 μ ± 1σ = 342 ~ 358。")
任務定義("2-5", _check_2_5, 提示="plt.title('拿鐵容量的常態分布')；下界 = mu - sigma。")

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import seaborn as sns, matplotlib.pyplot as plt
df = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0-rc.1/data/coffee_sales_aug.csv")

## 🎲 2-1　機率＝做很多很多次之後的比例
「擲骰子出現 6 的機率是 1/6」是什麼意思？不是「每 6 次一定有 1 次」，而是**擲很多很多次之後，6 出現的比例會越來越接近 1/6**。
電腦最擅長「做很多很多次」——這叫**模擬（simulation）**，這門課會一直用它來建立直覺。

In [ ]:
rng = np.random.default_rng(42)
for n in [10, 100, 1000, 100000]:
    骰子 = rng.integers(1, 7, size=n)
    print(f"擲 {n:>6} 次：6 出現的比例 = {(骰子 == 6).mean():.4f}")
print("理論值 1/6 =", round(1/6, 4))

## 2-2　常態分布：中間多、兩邊少的鐘形山丘
身高、體重、一杯拿鐵的容量……很多「很多小因素加起來」的量，都長成**常態分布（normal distribution）**：以平均為中心、左右對稱的鐘形。
只要知道**平均 μ** 和**標準差 σ**，整座山丘就決定了：

| 範圍 | 占比 |
|---|---|
| μ ± 1σ | 約 68% |
| μ ± 2σ | 約 95% |
| μ ± 3σ | 約 99.7% |

`scipy.stats.norm` 幫你算：`cdf(x)`＝「小於 x 的機率」（山丘左邊的面積）；`ppf(p)`＝反過來，「前 p 的位置在哪」。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
# 勇者咖啡的拿鐵：標示 350 ml，機器有誤差，實際容量 ~ N(350, 8)
mu, sigma = 350, 8
x = np.linspace(320, 380, 300)
plt.plot(x, stats.norm.pdf(x, mu, sigma))
plt.fill_between(x, stats.norm.pdf(x, mu, sigma), where=(x >= mu - sigma) & (x <= mu + sigma), alpha=.3, label="μ±1σ ≈ 68%")
plt.title("拿鐵容量的常態分布 N(350, 8)"); plt.xlabel("ml"); plt.legend(); plt.show()
print("容量 < 340 的機率", round(stats.norm.cdf(340, mu, sigma), 4))
print("340 ~ 360 之間的機率", round(stats.norm.cdf(360, mu, sigma) - stats.norm.cdf(340, mu, sigma), 4))
print("最大的 5% 從幾 ml 開始", round(stats.norm.ppf(0.95, mu, sigma), 1))

## 2-3　z 分數：把不同單位換成同一種貨幣
「這杯 365 ml 算多嗎？」要看它離平均**幾個標準差**：z = (x − μ) / σ。z = +1.9 代表比平均高 1.9 個標準差，是前 3% 的大杯。
把整欄資料都換成 z 分數叫**標準化（standardization）**：換完平均是 0、標準差是 1。
> 💱 為什麼機器學習要標準化？因為「公分」和「公斤」「元」的尺度不一樣，不換成同一種貨幣，模型會只看數字大的那一欄（Day 2 會親眼看到）。

In [ ]:
z = (365 - 350) / 8
print("365 ml 的 z 分數", round(z, 2), "→ 比它大的機率", round(1 - stats.norm.cdf(z), 4))
金額z = (df["金額"] - df["金額"].mean()) / df["金額"].std()
print("標準化後：平均", round(金額z.mean(), 6), "標準差", round(金額z.std(), 6))
print("金額 500 元的 z 分數", round((500 - df["金額"].mean()) / df["金額"].std(), 2), "→ 是很大的一單")

## 2-4　二項分布：n 個客人各有 p 的機會
今天有 10 位客人，每位有 30% 機率加購甜點，**恰好 3 位**加購的機率是多少？**至少 5 位**呢？
這種「n 次獨立、每次成功機率 p」的情境是**二項分布 Binomial(n, p)**：`binom.pmf(k, n, p)`＝恰好 k 次；`binom.cdf(k, n, p)`＝最多 k 次。期望值 = n × p。

In [ ]:
n, p = 10, 0.3
print("恰好 3 位", round(stats.binom.pmf(3, n, p), 4))
print("至少 5 位", round(1 - stats.binom.cdf(4, n, p), 4))
print("期望人數", n * p)
k = np.arange(0, 11)
plt.bar(k, stats.binom.pmf(k, n, p)); plt.title("10 位客人中加購甜點的人數 Binomial(10, 0.3)"); plt.xlabel("人數"); plt.show()

### 🎯 任務 2-1　用模擬算機率

用種子 42 的 `rng` 擲 10,000 次骰子存成 `骰子`，算出 `六點機率`（出現 6 的比例）；再擲**兩顆**骰子各 10,000 次（`a`、`b`），算出 `和為7機率`（a + b == 7 的比例）。

In [ ]:
# 🎯 任務 2-1　用模擬算機率（請保留這一行）
rng = np.random.default_rng(42)
骰子 = rng.integers(1, 7, size=10000)
六點機率 = ???
a = rng.integers(1, 7, size=10000)
b = rng.integers(1, 7, size=10000)
和為7機率 = ???
print(round(六點機率, 4), round(和為7機率, 4), "理論值都是", round(1/6, 4))

In [ ]:
檢查("2-1")   # ◀ 執行這一格，看看任務 2-1 有沒有過關

### 🎯 任務 2-2　常態分布的機率

拿鐵容量 ~ N(350, 8)。算出 `小於340機率`（`norm.cdf`）、`介於340到360機率`，以及 `前百分之五門檻`（容量最大的 5% 從幾 ml 開始，`norm.ppf(0.95, ...)`）。

In [ ]:
# 🎯 任務 2-2　常態分布的機率（請保留這一行）
mu, sigma = 350, 8
小於340機率 = ???
介於340到360機率 = ???
前百分之五門檻 = ???
print(round(小於340機率, 4), round(介於340到360機率, 4), round(前百分之五門檻, 2))

In [ ]:
檢查("2-2")   # ◀ 執行這一格，看看任務 2-2 有沒有過關

### 🎯 任務 2-3　z 分數與標準化

把金額整欄標準化存成 `金額z`（減平均、除以標準差），算出 `金額500的z`（金額為 500 元時的 z 分數，用公式算），並確認 `標準化平均`（`金額z.mean()`）接近 0、`標準化標準差` 接近 1。

In [ ]:
# 🎯 任務 2-3　z 分數與標準化（請保留這一行）
平均, 標準差 = df["金額"].mean(), df["金額"].std()
金額z = ???
金額500的z = ???
標準化平均 = 金額z.mean()
標準化標準差 = 金額z.std()
print(round(金額500的z, 3), round(標準化平均, 6), round(標準化標準差, 6))

In [ ]:
檢查("2-3")   # ◀ 執行這一格，看看任務 2-3 有沒有過關

### 🎯 任務 2-4　二項分布

10 位客人、每位 30% 加購甜點。算出 `恰好3人`（`binom.pmf`）、`至少5人`（1 − `binom.cdf(4, ...)`）、`期望人數`（n × p）。

In [ ]:
# 🎯 任務 2-4　二項分布（請保留這一行）
n, p = 10, 0.3
恰好3人 = ???
至少5人 = ???
期望人數 = ???
print(round(恰好3人, 4), round(至少5人, 4), 期望人數)

In [ ]:
檢查("2-4")   # ◀ 執行這一格，看看任務 2-4 有沒有過關

### 🎯 任務 2-5　畫出鐘形山丘

畫出 N(350, 8) 的機率密度曲線（x 從 320 到 380），標題要包含「常態」，並把 68% 範圍的 `下界`（μ − σ）與 `上界`（μ + σ）算出來。

In [ ]:
# 🎯 任務 2-5　畫出鐘形山丘（請保留這一行）
mu, sigma = 350, 8
x = np.linspace(320, 380, 300)
plt.plot(x, stats.norm.pdf(x, mu, sigma))
plt.title(???)
plt.show()
下界 = ???
上界 = ???
print(下界, 上界)

In [ ]:
檢查("2-5")   # ◀ 執行這一格，看看任務 2-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把八月每一天的營收加總（`df.groupby('日期')['金額'].sum()`）畫成直方圖，它比每筆金額更像常態嗎？為什麼？（提示：很多小因素加起來）
2. 用 `stats.norm.fit()` 幫每日營收配一條常態曲線，畫在直方圖上。

---
## 🔑 通關密語
　你已經能用機率和分布描述「通常會發生什麼」了。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🔬 L03 抽樣與檢定** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.0.0-rc.1/notebooks/L03_sampling_tests.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.1/